In [1]:
#!pip install pydantic-ai


# What is Pydantic?
- Pydantic is a Python library for data validation and settings management using Python type annotations.

- Think of it like how statically-typed languages or databases enforce types on variables.

- It allows for robust error handling by validating data at runtime and raising helpful errors when data doesn't match the expected schema.

- It makes code more self-descriptive, since you explicitly define the shape and types of data your functions and classes expect.

- The trade-off is that code can become a bit more verbose, especially when defining complex models

In [3]:
from pydantic import BaseModel, ConfigDict

# User model with string length validation
class User(BaseModel):
    model_config = ConfigDict(str_max_length=10)

    id: int
    name: str

class Phone(BaseModel):
    id: int
    phone_number: str

class Contact(BaseModel):
    id: int
    user: User
    phone: Phone

# Create instances
user = User(id=1, name="Tom")
phone = Phone(id=1, phone_number="111-111-1111")
tom = Contact(id=1, user=user, phone=phone)

print(tom)

user2 = Contact(id=2, user=User(id=2,name='lo'),phone=Phone(id=2,phone_number='111-111-1111'))



id=1 user=User(id=1, name='Tom') phone=Phone(id=1, phone_number='111-111-1111')


# Unions are the complex part
- Unless you plan ahead then its fine

In [6]:
from typing import Union, List

class Employee(BaseModel):
  id: int
  user: User
  phone: Phone
  pay_rate: int

class Student(BaseModel):
  id: int
  user: User
  phone: Phone
  gpa: float


class UserDirectory(BaseModel):
  id: int
  users: List[Union[Employee, Student]]


In [8]:
directory = UserDirectory(
    id=1,
    users=[
        Employee(
            id=101,
            user=User(id=1, name="Alice"),
            phone=Phone(id=11, phone_number="123-456"),
            pay_rate=30
        ),
        Student(
            id=202,
            user=User(id=2, name="Bob"),
            phone=Phone(id=22, phone_number="789-012"),
            gpa=3.9
        )
    ]
)

print(directory)

id=1 users=[Employee(id=101, user=User(id=1, name='Alice'), phone=Phone(id=11, phone_number='123-456'), pay_rate=30), Student(id=202, user=User(id=2, name='Bob'), phone=Phone(id=22, phone_number='789-012'), gpa=3.9)]


Pyandatic AI
- Agentic Library
- Similar to langchain


In [12]:
# This code is needed to run agents in colab, but if you were using python normally wouldn't be needed
import nest_asyncio

nest_asyncio.apply()

from pydantic_ai import Agent
# pydantic-ai
from pydantic_ai import Agent, RunContext
from pydantic_ai.models.gemini import GeminiModel
from pydantic_ai.providers.google_gla import GoogleGLAProvider

model = GeminiModel(
    'gemini-2.0-flash',
        provider=GoogleGLAProvider(api_key='')
)
agent = Agent(
    model,
    system_prompt='Be concise, reply with one sentence.',
)
result = agent.run_sync('Where does "hello world" come from?')
print(result.output)

"Hello, World!" originated as an example program in Brian Kernighan and Dennis Ritchie's 1972 book "A Tutorial Introduction to the Language B."



# This code demostrates tool use

In [14]:
from dataclasses import dataclass
from typing import Optional

from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext


@dataclass
class InfoExtractionDependencies:
    # Add any external systems or context later if needed
    pass


class UserInfo(BaseModel):
    name: Optional[str] = Field(description="The user's full name, if available")
    email: Optional[str] = Field(description="The user's email address, if available")
    user_id: Optional[int] = Field(description="The user's ID number, if stated")


info_extraction_agent = Agent(
    model,  # or any other supported model
    deps_type=InfoExtractionDependencies,
    output_type=UserInfo,
    system_prompt=(
        "Extract the user's name, email, and ID from their message. "
        "If any of them are missing, return None for that field."
    ),
)



text = "Hi, I’m Jane Doe. My email is jane@example.com and my ID is 823."
result = await info_extraction_agent.run(text, deps=InfoExtractionDependencies())
print(result.output)

# Output:
# name='Jane Doe' email='jane@example.com' user_id=823


name='Jane Doe' email='jane@example.com' user_id=823


# The Following code demostrates using multiple agents


In [16]:
from dataclasses import dataclass
from typing import Optional, Dict
from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext
from pydantic_ai.usage import UsageLimits



@dataclass
class DirectoryDeps:
    directory: UserDirectory


user_extraction_agent = Agent(
    model,
    deps_type=DirectoryDeps,
    output_type=Union[Employee, Student],
    system_prompt=(
        "You are given a text message describing a person. "
        "Extract either an Employee or a Student and return the structured object."
    ),
)


user_saving_agent = Agent(
    model,
    deps_type=DirectoryDeps,
    system_prompt="Use the `store_user` tool to save a user to the directory."
)

@user_saving_agent.tool
def store_user(ctx: RunContext[DirectoryDeps], user: Union[Employee, Student]) -> str:
    ctx.deps.directory.users.append(user)
    return f"Saved {user.user.name} as {'Employee' if isinstance(user, Employee) else 'Student'}"


def run_pipeline():
    # Set up shared memory/dependencies
    directory = UserDirectory(id=1,users=[])
    deps = DirectoryDeps(directory=directory)

    # Extract user from text
    extraction_result = user_extraction_agent.run_sync(
        "My name is Jane Doe, I'm a student with ID 42. My GPA is 3.8. My number is 555-1212.",
        deps=deps
    )
    print("Extracted:", extraction_result.output)

    # Save that user via second agent
    saving_result = user_saving_agent.run_sync(
        f"Save this: {extraction_result.output.model_dump_json()}",
        deps=deps
    )
    print("Saved:", saving_result.output)

    # Check directory
    print("\nUserDirectory contents:")
    for u in deps.directory.users:
        print(u)


In [18]:
run_pipeline()

Extracted: id=42 user=User(id=42, name='Jane Doe') phone=Phone(id=42, phone_number='555-1212') gpa=3.8
Saved: OK. I've saved Jane Doe as a student.

UserDirectory contents:
id=42 user=User(id=42, name='Jane Doe') phone=Phone(id=42, phone_number='555-1212') gpa=3.8


# Message history
- This part covers https://ai.pydantic.dev/message-history/

In [20]:
from dataclasses import dataclass
from typing import Optional, Dict, Union, List
from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext
from pydantic_ai.messages import ModelMessage

# Data Models
class User(BaseModel):
    name: str
    phone: Optional[str] = None

class Employee(BaseModel):
    user: User
    employee_id: int
    department: str

class Student(BaseModel):
    user: User
    student_id: int
    gpa: float

class UserDirectory(BaseModel):
    id: int
    users: List[Union[Employee, Student]] = Field(default_factory=list)

@dataclass
class DirectoryDeps:
    directory: UserDirectory


# Agent 1: Extract user information
user_extraction_agent = Agent(
    model,
    deps_type=DirectoryDeps,
    output_type=Union[Employee, Student],
    system_prompt=(
        "You are an expert at extracting user information from text. "
        "Analyze the message and determine if the person is an Employee or Student. "
        "Extract all relevant information and return the appropriate structured object. "
        "For employees, look for department and employee ID. "
        "For students, look for student ID and GPA."
    ),
)

# Agent 2: Save user to directory
user_saving_agent = Agent(
    model,
    deps_type=DirectoryDeps,
    system_prompt=(
        "You are responsible for saving users to the directory. "
        "Use the `store_user` tool to save users and provide helpful feedback."
    )
)

@user_saving_agent.tool
def store_user(ctx: RunContext[DirectoryDeps], user: Union[Employee, Student]) -> str:
    ctx.deps.directory.users.append(user)
    user_type = 'Employee' if isinstance(user, Employee) else 'Student'
    return f"Successfully saved {user.user.name} as {user_type}. Directory now contains {len(ctx.deps.directory.users)} users."

# Agent 3: Query and analyze directory
directory_query_agent = Agent(
    model,
    deps_type=DirectoryDeps,
    system_prompt=(
        "You help users query and analyze the user directory. "
        "Use the provided tools to search and analyze user data."
    )
)

@directory_query_agent.tool
def search_users(ctx: RunContext[DirectoryDeps], name_pattern: str = "") -> str:
    """Search for users by name pattern"""
    matching_users = []
    for user in ctx.deps.directory.users:
        if name_pattern.lower() in user.user.name.lower():
            user_type = 'Employee' if isinstance(user, Employee) else 'Student'
            matching_users.append(f"{user.user.name} ({user_type})")

    if matching_users:
        return f"Found {len(matching_users)} matching users: {', '.join(matching_users)}"
    else:
        return f"No users found matching '{name_pattern}'"

@directory_query_agent.tool
def get_directory_stats(ctx: RunContext[DirectoryDeps]) -> str:
    """Get statistics about the directory"""
    total_users = len(ctx.deps.directory.users)
    employees = sum(1 for u in ctx.deps.directory.users if isinstance(u, Employee))
    students = sum(1 for u in ctx.deps.directory.users if isinstance(u, Student))

    return f"Directory contains {total_users} total users: {employees} employees, {students} students"

def run_enhanced_pipeline():
    """Enhanced pipeline demonstrating message history usage"""

    # Set up shared directory
    directory = UserDirectory(id=1, users=[])
    deps = DirectoryDeps(directory=directory)

    print("=== Enhanced User Directory Pipeline ===\n")

    # Step 1: Extract first user
    print("Step 1: Extracting first user...")
    result1 = user_extraction_agent.run_sync(
        "Hi! I'm John Smith, employee #12345 in the Engineering department. You can reach me at 555-0123.",
        deps=deps
    )
    print(f"Extracted: {result1.output}")

    # Step 2: Save first user with conversation context
    print("\nStep 2: Saving first user...")
    result2 = user_saving_agent.run_sync(
        f"Please save this user: {result1.output.model_dump_json()}",
        deps=deps,
        message_history=result1.new_messages()  # Using message history!
    )
    print(f"Save result: {result2.output}")

    # Step 3: Extract second user, building on conversation
    print("\nStep 3: Extracting second user...")
    result3 = user_extraction_agent.run_sync(
        "Now I have another person: Sarah Johnson, student ID 67890, GPA 3.9, phone 555-0456",
        deps=deps,
        message_history=result2.all_messages()  # Continue conversation
    )
    print(f"Extracted: {result3.output}")

    # Step 4: Save second user
    print("\nStep 4: Saving second user...")
    result4 = user_saving_agent.run_sync(
        f"Save this student: {result3.output.model_dump_json()}",
        deps=deps,
        message_history=result3.new_messages()
    )
    print(f"Save result: {result4.output}")

    # Step 5: Query the directory with conversation context
    print("\nStep 5: Querying directory...")
    result5 = directory_query_agent.run_sync(
        "What's in our directory now? Give me the stats and search for anyone named 'John'.",
        deps=deps,
        message_history=result4.all_messages()  # Full conversation context
    )
    print(f"Query result: {result5.output}")

    # Step 6: Show message history analysis
    print("\n=== Message History Analysis ===")
    print(f"Total messages in final conversation: {len(result5.all_messages())}")
    print(f"Messages from last run only: {len(result5.new_messages())}")

    # Step 7: Show directory contents
    print("\n=== Final Directory Contents ===")
    for i, user in enumerate(deps.directory.users, 1):
        user_type = 'Employee' if isinstance(user, Employee) else 'Student'
        print(f"{i}. {user.user.name} ({user_type})")
        if isinstance(user, Employee):
            print(f"   Department: {user.department}, ID: {user.employee_id}")
        else:
            print(f"   Student ID: {user.student_id}, GPA: {user.gpa}")

def demonstrate_message_serialization():
    """Demonstrate message serialization as shown in the docs"""
    from pydantic_core import to_jsonable_python
    from pydantic_ai.messages import ModelMessagesTypeAdapter

    directory = UserDirectory(id=1, users=[])
    deps = DirectoryDeps(directory=directory)

    print("\n=== Message Serialization Demo ===")

    # Run an agent
    result = user_extraction_agent.run_sync(
        "I'm Alice Cooper, student ID 11111, GPA 4.0",
        deps=deps
    )

    # Get messages
    messages = result.all_messages()
    print(f"Original messages: {len(messages)} messages")

    # Serialize to JSON-compatible format
    serializable_messages = to_jsonable_python(messages)
    print("Messages serialized to JSON-compatible format ✓")

    # Deserialize back
    restored_messages = ModelMessagesTypeAdapter.validate_python(serializable_messages)
    print(f"Messages restored: {len(restored_messages)} messages")

    # Use restored messages in new run
    result2 = user_extraction_agent.run_sync(
        "Now tell me about Bob Wilson, employee 22222 in Sales",
        deps=deps,
        message_history=restored_messages
    )
    print(f"Successfully used restored messages in new run ✓")

# Example usage for students
if __name__ == "__main__":
    # Run the main pipeline
    run_enhanced_pipeline()

    # Demonstrate message serialization
    demonstrate_message_serialization()



=== Enhanced User Directory Pipeline ===

Step 1: Extracting first user...
Extracted: user=User(name='John Smith', phone='555-0123') employee_id=12345 department='Engineering'

Step 2: Saving first user...
Save result: Okay, I have stored the employee information for John Smith.


Step 3: Extracting second user...
Extracted: user=User(name='Sarah Johnson', phone='555-0456') student_id=67890 gpa=3.9

Step 4: Saving second user...
Save result: OK. I have stored Sarah Johnson's information. Anything else?


Step 5: Querying directory...
Query result: The directory contains 1 user: 1 student. I found one user matching "John": Sarah Johnson (Student).


=== Message History Analysis ===
Total messages in final conversation: 11
Messages from last run only: 4

=== Final Directory Contents ===
1. Sarah Johnson (Student)
   Student ID: 67890, GPA: 3.9

=== Message Serialization Demo ===
Original messages: 3 messages
Messages serialized to JSON-compatible format ✓
Messages restored: 3 messages
Su

# General Output
- This part covers https://ai.pydantic.dev/output/

In [26]:
"""
PydanticAI Output Examples Lab
Demonstrates different output types and validation patterns
"""

from dataclasses import dataclass
from typing import Union, Optional, List
from datetime import date
from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext, ModelRetry
import asyncio
# Long example, renter api key, if you ran this in one go it may throw limit error
model = GeminiModel(
    'gemini-2.0-flash',
        provider=GoogleGLAProvider(api_key='')
)

# =============================================================================
# Example 1: Union Output Types - Smart Form Processing
# =============================================================================

class PersonInfo(BaseModel):
    """Complete person information"""
    name: str
    age: int
    email: str
    phone: Optional[str] = None

class IncompleteData(BaseModel):
    """When data is missing or unclear"""
    missing_fields: List[str]
    message: str

# Agent that returns either complete data or explains what's missing
form_processor = Agent(
    model,
    output_type=Union[PersonInfo, IncompleteData],  # type: ignore
    system_prompt=(
        "Extract person information from text. If any required fields "
        "(name, age, email) are missing or unclear, return IncompleteData "
        "explaining what's needed."
    )
)

def demo_union_outputs():
    """Demonstrate union output types"""
    print("=== Union Output Types Demo ===")

    # Complete data
    result1 = form_processor.run_sync(
        "Hi, I'm Sarah Johnson, 25 years old, email sarah@example.com, phone 555-1234"
    )
    print(f"Complete data: {result1.output}")
    print(f"Type: {type(result1.output).__name__}")

    # Incomplete data
    result2 = form_processor.run_sync(
        "My name is Bob and I'm 30"
    )
    print(f"Incomplete data: {result2.output}")
    print(f"Type: {type(result2.output).__name__}")

# =============================================================================
# Example 2: Output Functions - Data Processing Pipeline
# =============================================================================

class DataValidationError(BaseModel):
    """Represents a validation failure"""
    error_type: str
    message: str

@dataclass
class ProcessingDeps:
    processed_count: int = 0

def process_student_record(
    ctx: RunContext[ProcessingDeps],
    name: str,
    student_id: int,
    gpa: float
) -> dict:
    """Process a student record with validation"""

    # Validate student ID format
    if student_id < 10000 or student_id > 99999:
        raise ModelRetry("Student ID must be 5 digits (10000-99999)")

    # Validate GPA range
    if not (0.0 <= gpa <= 4.0):
        raise ModelRetry("GPA must be between 0.0 and 4.0")

    # Validate name
    if len(name.strip()) < 2:
        raise ModelRetry("Name must be at least 2 characters")

    # Success - increment counter and return processed data
    ctx.deps.processed_count += 1
    return {
        "name": name.strip().title(),
        "student_id": student_id,
        "gpa": round(gpa, 2),
        "status": "processed",
        "record_number": ctx.deps.processed_count
    }

# Agent with output function
student_processor = Agent(
    model,
    deps_type=ProcessingDeps,
    output_type=[process_student_record, DataValidationError],
    system_prompt=(
        "Extract student information and process it. "
        "Look for name, student ID (5 digits), and GPA (0.0-4.0). "
        "If data is invalid or missing, return DataValidationError."
    )
)

def demo_output_functions():
    """Demonstrate output functions with validation"""
    print("\n=== Output Functions Demo ===")

    deps = ProcessingDeps()

    # Valid student record
    result1 = student_processor.run_sync(
        "Student: alice smith, ID: 12345, GPA: 3.85",
        deps=deps
    )
    print(f"Processed: {result1.output}")

    # Invalid GPA (will trigger ModelRetry and correction)
    result2 = student_processor.run_sync(
        "Student: bob jones, ID: 67890, GPA: 5.2",  # Invalid GPA
        deps=deps
    )
    print(f"Result after validation: {result2.output}")

    # Missing data
    result3 = student_processor.run_sync(
        "Just a student named Charlie",
        deps=deps
    )
    print(f"Missing data result: {result3.output}")

# =============================================================================
# Example 3: Output Validators - Real-time Data Validation
# =============================================================================

class EmailData(BaseModel):
    recipient: str
    subject: str
    body: str

class EmailError(BaseModel):
    error_message: str

# Simulated email validation service
def validate_email_address(email: str) -> bool:
    """Simulate email validation (normally would be async API call)"""
    # Simple validation - in reality this might call an API
    return "@" in email and "." in email.split("@")[1]

def check_spam_content(subject: str, body: str) -> bool:
    """Simulate spam detection"""
    spam_words = ["urgent", "click now", "free money", "limited time"]
    content = f"{subject} {body}".lower()
    return any(word in content for word in spam_words)

email_agent = Agent(
    model,
    output_type=Union[EmailData, EmailError],  # type: ignore
    system_prompt=(
        "Create professional email content based on user input. "
        "Extract recipient, subject, and body text."
    )
)

@email_agent.output_validator
def validate_email_output(ctx: RunContext, output: Union[EmailData, EmailError]) -> Union[EmailData, EmailError]:
    """Validate email before sending"""
    if isinstance(output, EmailError):
        return output

    # Validate email address
    if not validate_email_address(output.recipient):
        raise ModelRetry(f"Invalid email address: {output.recipient}")

    # Check for spam content
    if check_spam_content(output.subject, output.body):
        raise ModelRetry("Content appears spammy. Please use more professional language.")

    # Check length limits
    if len(output.subject) > 100:
        raise ModelRetry("Subject line too long (max 100 characters)")

    return output

def demo_output_validators():
    """Demonstrate output validators"""
    print("\n=== Output Validators Demo ===")

    # Valid email
    result1 = email_agent.run_sync(
        "Send email to john@company.com about quarterly meeting on Friday"
    )
    print(f"Valid email: {result1.output}")

    # Invalid email (will trigger validation and retry)
    result2 = email_agent.run_sync(
        "Send urgent message to badaddress about free money opportunity - click now!"
    )
    print(f"After validation: {result2.output}")

# =============================================================================
# Example 4: Streaming Structured Output
# =============================================================================

from typing_extensions import TypedDict

class StudentProfile(TypedDict, total=False):
    name: str
    student_id: int
    major: str
    year: int
    gpa: float
    activities: str

profile_agent = Agent(
    model,
    output_type=StudentProfile,
    system_prompt="Extract a complete student profile from the input text."
)

async def demo_streaming_output():
    """Demonstrate streaming structured output"""
    print("\n=== Streaming Output Demo ===")

    user_input = (
        "I'm Jessica Martinez, student ID 54321, Computer Science major, "
        "junior year, 3.7 GPA. I'm involved in robotics club and debate team."
    )

    print("Streaming profile as it builds:")
    final_profile = None
    async with profile_agent.run_stream(user_input) as result:
        async for profile in result.stream():
            print(f"  {profile}")
            final_profile = profile

    print("Final complete profile:")
    print(f"  {final_profile}")

# =============================================================================
# Example 5: Multi-Stage Processing with Different Output Types
# =============================================================================

class TaskAssignment(BaseModel):
    task_id: str
    assignee: str
    priority: str
    due_date: str

class TaskError(BaseModel):
    error_type: str
    suggestion: str

def route_to_manager(ctx: RunContext, task: str, urgency: str) -> str:
    """Route urgent tasks to management"""
    if urgency.lower() in ["high", "urgent", "critical"]:
        return f"ESCALATED TO MANAGER: {task}"
    else:
        raise ModelRetry("This task doesn't need manager escalation. Process normally.")

# Multi-output agent that can create tasks, escalate, or report errors
task_router = Agent(
    model,
    output_type=[TaskAssignment, route_to_manager, TaskError],
    system_prompt=(
        "Process task requests. Create TaskAssignment for normal tasks, "
        "use route_to_manager for urgent items, or TaskError for problems."
    )
)

def demo_multi_output_types():
    """Demonstrate multiple output types in one agent"""
    print("\n=== Multi-Output Types Demo ===")

    # Normal task
    result1 = task_router.run_sync(
        "Assign code review task to Sarah for next Friday, medium priority"
    )
    print(f"Normal task: {result1.output}")

    # Urgent escalation
    result2 = task_router.run_sync(
        "URGENT: Server is down, need immediate attention!"
    )
    print(f"Escalated: {result2.output}")

    # Error case
    result3 = task_router.run_sync(
        "Do something with the thing"
    )
    print(f"Error handling: {result3.output}")

# =============================================================================
# Main Demo Runner
# =============================================================================

async def run_all_demos():
    """Run all demonstrations"""
    print("PydanticAI Output Examples Lab")
    print("=" * 50)

    # Sync demos
    demo_union_outputs()
    demo_output_functions()
    demo_output_validators()
    demo_multi_output_types()

    # Async demo
    await demo_streaming_output()
    # Uncomment to run all demos:
asyncio.run(run_all_demos())

PydanticAI Output Examples Lab
=== Union Output Types Demo ===
Complete data: name='Sarah Johnson' age=25 email='sarah@example.com' phone='555-1234'
Type: PersonInfo
Incomplete data: missing_fields=['email'] message='Missing email'
Type: IncompleteData

=== Output Functions Demo ===
Processed: {'name': 'Alice Smith', 'student_id': 12345, 'gpa': 3.85, 'status': 'processed', 'record_number': 1}
Result after validation: error_type='DataValidationError' message='GPA is invalid'
Missing data result: error_type='MissingData' message='Missing student ID and GPA'

=== Output Validators Demo ===
Valid email: recipient='john@company.com' subject='Quarterly Meeting' body='Quarterly meeting on Friday'
After validation: error_message='Invalid email address: badaddress\n\nFix the errors and try again. '

=== Multi-Output Types Demo ===
Normal task: task_id='code review' assignee='Sarah' priority='medium' due_date='next Friday'
Escalated: ESCALATED TO MANAGER: Server down
Error handling: error_type='

# PydanticAI Concepts Q&A Guide

## Messages and Chat History

**Q: How do I maintain conversation context between agent runs?**

A: Use the `message_history` parameter in `agent.run_sync()` or `agent.run()`. You can access messages from previous runs using:
- `result.all_messages()` - includes messages from prior runs
- `result.new_messages()` - only messages from current run

```python
result1 = agent.run_sync("Tell me a joke")
result2 = agent.run_sync("Explain it", message_history=result1.new_messages())
```

**Q: Can I save and restore conversation history?**

A: Yes! Use `ModelMessagesTypeAdapter` for serialization:

```python
from pydantic_core import to_jsonable_python
from pydantic_ai.messages import ModelMessagesTypeAdapter

# Serialize
serializable = to_jsonable_python(result.all_messages())
# Restore
restored = ModelMessagesTypeAdapter.validate_python(serializable)
```

**Q: What's the difference between StreamedRunResult and RunResult messages?**

A: With `StreamedRunResult`, messages are only complete after the stream finishes. You must await one of: `stream()`, `stream_text()`, `stream_structured()`, or `get_output()` before accessing complete messages.

**Q: Can I use messages across different models?**

A: Yes! Messages are model-independent. You can use messages from an OpenAI agent run in a Google Gemini agent run.

## Output Types and Validation

**Q: What types can I use for `output_type`?**

A: Several options:
- **Plain text**: `str` (default)
- **Pydantic models**: `MyModel`
- **Union types**: `Union[Model1, Model2]` or `Model1 | Model2`
- **Lists**: `[Model1, Model2]` (equivalent to union)
- **Output functions**: Custom functions that process and validate

**Q: How do union output types work?**

A: The agent chooses the appropriate type based on context. Each type becomes a separate tool:

```python
class Success(BaseModel):
    data: str

class Error(BaseModel):
    message: str

agent = Agent(model, output_type=Union[Success, Error])
# Agent decides which to return based on input
```

**Q: What are output functions and when should I use them?**

A: Output functions let you process and validate model responses before returning results. Use them when you need:
- Custom validation logic
- Data transformation
- Integration with external systems
- Stateful processing

```python
def process_data(ctx: RunContext, raw_data: str) -> ProcessedData:
    if not validate(raw_data):
        raise ModelRetry("Invalid data format")
    return ProcessedData(cleaned=clean(raw_data))

agent = Agent(model, output_type=process_data)
```

**Q: How do output validators differ from output functions?**

A: **Output validators** run after the output is generated and can trigger retries:

```python
@agent.output_validator
def validate_output(ctx: RunContext, output: MyModel) -> MyModel:
    if not is_valid(output):
        raise ModelRetry("Output failed validation")
    return output
```

**Output functions** ARE the output - they process model arguments into the final result.

**Q: When should I use ModelRetry?**

A: Use `ModelRetry` in output functions or validators when:
- Data validation fails
- External API calls fail
- Business logic requirements aren't met
- You want the model to try again with feedback

The model receives your retry message and attempts to correct the issue.

## Streaming

**Q: How do I stream text responses?**

A: Use `agent.run_stream()` with `stream_text()`:

```python
async with agent.run_stream("Write a story") as result:
    async for text in result.stream_text():
        print(text)  # Incremental text
    
    # Or for deltas only:
    async for delta in result.stream_text(delta=True):
        print(delta)  # Just new characters
```

**Q: Can I stream structured output?**

A: Yes! Use TypedDict for best compatibility:

```python
class UserProfile(TypedDict, total=False):
    name: str
    age: int
    email: str

async with agent.run_stream(input_text) as result:
    async for profile in result.stream():
        print(profile)  # Partially complete profiles
```

**Q: How do I handle validation errors during streaming?**

A: Use `stream_structured()` with manual validation:

```python
async with agent.run_stream(input_text) as result:
    async for message, is_last in result.stream_structured():
        try:
            validated = await result.validate_structured_output(
                message, allow_partial=not is_last
            )
            print(validated)
        except ValidationError:
            continue  # Skip invalid partial states
```

## Dependencies and Context

**Q: How do I share state between agents?**

A: Use dependencies with `deps_type`:

```python
@dataclass
class SharedState:
    database: Database
    user_count: int = 0

agent = Agent(model, deps_type=SharedState)

# Pass same deps instance to multiple agents
deps = SharedState(database=db)
result1 = agent1.run_sync("query", deps=deps)
result2 = agent2.run_sync("update", deps=deps)
```

**Q: What can I access in RunContext?**

A: `RunContext` provides:
- `ctx.deps` - Your dependency object
- `ctx.messages` - Current conversation messages
- `ctx.usage` - Token/request usage information

**Q: How do I pass context between tools and output functions?**

A: All tools and output functions in the same agent run share the same `RunContext`, so they access the same `deps` object:

```python
@agent.tool
def gather_data(ctx: RunContext[MyDeps]) -> str:
    ctx.deps.collected_data.append("new data")
    return "Data collected"

@agent.output_function  
def process_results(ctx: RunContext[MyDeps]) -> Results:
    # Access data collected by tools
    return Results(data=ctx.deps.collected_data)
```

## Multi-Agent Patterns

**Q: How do I create agent pipelines?**

A: Chain agents using message history and shared dependencies:

```python
# Agent 1: Extract data
result1 = extraction_agent.run_sync(input_text, deps=shared_deps)

# Agent 2: Process data with context
result2 = processing_agent.run_sync(
    f"Process: {result1.output}",
    deps=shared_deps,
    message_history=result1.new_messages()
)

# Agent 3: Final output with full context  
result3 = output_agent.run_sync(
    "Generate report",
    deps=shared_deps,
    message_history=result2.all_messages()
)
```

**Q: How do I handle agent handoffs?**

A: Use output functions to route between agents:

```python
async def hand_off_to_specialist(ctx: RunContext, task: str) -> Results:
    messages = ctx.messages[:-1]  # Exclude current tool call
    
    result = await specialist_agent.run(task, message_history=messages)
    
    if isinstance(result.output, ErrorType):
        raise ModelRetry(f"Specialist failed: {result.output.message}")
    
    return result.output
```

**Q: What's the best way to share data between agents?**

A: Use a combination of:
1. **Shared dependencies** for state/resources
2. **Message history** for conversation context  
3. **Structured outputs** for data transfer
4. **External storage** for persistence (database, files)

## Best Practices

**Q: When should I use streaming vs regular runs?**

A: Use streaming for:
- Long responses where users want progressive feedback
- User interfaces that show real-time progress
- Processing large amounts of data incrementally

Use regular runs for:
- Simple request/response patterns
- When you need the complete result immediately
- Batch processing scenarios

**Q: How do I debug agent interactions?**

A: Access detailed information from results:
- `result.usage()` - Token and request counts
- `result.all_messages()` - Full conversation history
- Exception handling around `ModelRetry` scenarios
- Logging in tools and output functions

**Q: What's the recommended pattern for error handling?**

A: Use a layered approach:
1. **Input validation** in tools/output functions with `ModelRetry`
2. **Business logic errors** as structured output types
3. **System errors** with proper exception handling
4. **User-friendly error messages** in error output types

```python
class Success(BaseModel):
    result: str

class BusinessError(BaseModel):
    error_type: str
    user_message: str
    
class SystemError(BaseModel):
    technical_details: str

agent = Agent(model, output_type=Union[Success, BusinessError, SystemError])
```